In [ ]:
import os
import json
import random
import hashlib
from typing import Dict, List, Optional

from transformers import AutoTokenizer

from datasets import load_dataset

/workspace/venvs/qwen/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# =========================================================
# Config
# =========================================================
MODEL_NAME = "meta-llama/Llama-3.1-8B"   # base model tokenizer
OUT_DIR = "/root/data/sft_data"
SEED = 0

N_SFT = 30_000
N_CALIB = 1_000

USE_HF_DATASET = True
HF_DATASET_ID = "yahma/alpaca-cleaned"            

# ---------- 长度约束 ----------
# SFT：偏训练友好
SFT_MIN_TOTAL_TOK = 32
SFT_MAX_TOTAL_TOK = 768
SFT_MIN_TARGET_TOK = 8
SFT_MAX_TARGET_TOK = 256

# calib：覆盖更宽一点
CALIB_MIN_TOTAL_TOK = 32
CALIB_MAX_TOTAL_TOK = 1024
CALIB_MIN_TARGET_TOK = 4
CALIB_MAX_TARGET_TOK = 384

CALIB_BUCKETS = {
    "short": {"min": 32,  "max": 128,  "target": 300},
    "mid":   {"min": 129, "max": 384,  "target": 500},
    "long":  {"min": 385, "max": 1024, "target": 200},
}
assert sum(v["target"] for v in CALIB_BUCKETS.values()) == N_CALIB

In [ ]:
# =========================================================
# Utils
# =========================================================
def set_seed(seed: int):
    random.seed(seed)

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def write_jsonl(path: str, records: List[Dict]):
    ensure_dir(os.path.dirname(path))
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def sha1_text(x: str) -> str:
    return hashlib.sha1(x.encode("utf-8")).hexdigest()

def normalize_text(x: Optional[str]) -> str:
    if x is None:
        return ""
    x = str(x).strip()
    return x

In [ ]:
def build_prompt_text(instruction: str, input_text: str) -> str:
    """
    用统一的简单模板，不走 chat template。
    因为这版目标是 base model + Alpaca，先跑通并控制成本。
    """
    instruction = normalize_text(instruction)
    input_text = normalize_text(input_text)

    if input_text:
        return (
            "Below is an instruction that describes a task, paired with an input "
            "that provides further context. Write a response that appropriately completes the request.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            "### Response:\n"
        )
    else:
        return (
            "Below is an instruction that describes a task. "
            "Write a response that appropriately completes the request.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            "### Response:\n"
        )

def get_text_len(tok, text: str) -> int:
    return len(tok(text, add_special_tokens=False)["input_ids"])

def load_alpaca_examples() -> List[Dict]:
    if USE_HF_DATASET:
        from datasets import load_dataset
        ds = load_dataset(HF_DATASET_ID, split="train")
        return [ds[i] for i in range(len(ds))]

    raise ValueError("Please enable USE_HF_DATASET.")

def build_record(ex: Dict, tok, source_idx: int) -> Optional[Dict]:
    instruction = normalize_text(ex.get("instruction"))
    input_text = normalize_text(ex.get("input"))
    output_text = normalize_text(ex.get("output"))

    if not instruction or not output_text:
        return None

    prompt_text = build_prompt_text(instruction, input_text)
    target_text = output_text

    prompt_len = get_text_len(tok, prompt_text)
    target_len = get_text_len(tok, target_text)
    total_len = get_text_len(tok, prompt_text + target_text)

    # 统一转成 messages 结构，后面 SFT / KD 更顺手
    user_content = instruction if not input_text else f"{instruction}\n\nInput:\n{input_text}"
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": target_text},
    ]

    uid = sha1_text(
        f"{source_idx}|||{instruction[:300]}|||{input_text[:300]}|||{output_text[:300]}"
    )

    return {
        "id": uid,
        "messages": messages,
        "prompt_text": prompt_text,
        "target_text": target_text,
        "meta": {
            "source_index": source_idx,
            "prompt_tokens": prompt_len,
            "target_tokens": target_len,
            "total_tokens": total_len,
            "has_input": bool(input_text),
        }
    }

def sft_accept(rec: Dict) -> bool:
    m = rec["meta"]
    return (
        SFT_MIN_TOTAL_TOK <= m["total_tokens"] <= SFT_MAX_TOTAL_TOK and
        SFT_MIN_TARGET_TOK <= m["target_tokens"] <= SFT_MAX_TARGET_TOK
    )

def calib_bucket_name(total_tokens: int) -> Optional[str]:
    for name, cfg in CALIB_BUCKETS.items():
        if cfg["min"] <= total_tokens <= cfg["max"]:
            return name
    return None

def calib_accept(rec: Dict) -> Optional[str]:
    m = rec["meta"]
    if not (
        CALIB_MIN_TOTAL_TOK <= m["total_tokens"] <= CALIB_MAX_TOTAL_TOK and
        CALIB_MIN_TARGET_TOK <= m["target_tokens"] <= CALIB_MAX_TARGET_TOK
    ):
        return None
    return calib_bucket_name(m["total_tokens"])

In [ ]:
# =========================================================
# Main
# =========================================================
set_seed(SEED)
ensure_dir(OUT_DIR)

print(f"Loading tokenizer: {MODEL_NAME}")
tok = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    trust_remote_code=True
)

print("Loading Alpaca data...")
examples = load_alpaca_examples()
print(f"Loaded examples: {len(examples)}")

idxs = list(range(len(examples)))
random.shuffle(idxs)

sft_records: List[Dict] = []
calib_buckets: Dict[str, List[Dict]] = {k: [] for k in CALIB_BUCKETS}

seen_sft = set()
seen_calib = set()

def calib_done() -> bool:
    return all(len(calib_buckets[k]) >= CALIB_BUCKETS[k]["target"] for k in CALIB_BUCKETS)

for n_scanned, i in enumerate(idxs, start=1):
    ex = examples[i]
    rec = build_record(ex, tok, source_idx=i)
    if rec is None:
        continue

    dup_key = sha1_text(
        rec["prompt_text"][:500] + "|||" + rec["target_text"][:500]
    )

    # ---------- SFT ----------
    if len(sft_records) < N_SFT and dup_key not in seen_sft and sft_accept(rec):
        sft_records.append(rec)
        seen_sft.add(dup_key)

    # ---------- calib ----------
    bucket = calib_accept(rec)
    if bucket is not None:
        if (
            len(calib_buckets[bucket]) < CALIB_BUCKETS[bucket]["target"]
            and dup_key not in seen_calib
        ):
            calib_buckets[bucket].append(rec)
            seen_calib.add(dup_key)

    if len(sft_records) >= N_SFT and calib_done():
        break

    if n_scanned % 5000 == 0:
        calib_count = sum(len(v) for v in calib_buckets.values())
        print(
            f"scanned={n_scanned} | "
            f"sft={len(sft_records)}/{N_SFT} | "
            f"calib={calib_count}/{N_CALIB} | "
            f"buckets={{"
            + ", ".join(f'{k}:{len(v)}/{CALIB_BUCKETS[k]["target"]}' for k, v in calib_buckets.items())
            + "}"
        )

calib_records = []
for k in ["short", "mid", "long"]:
    calib_records.extend(calib_buckets[k])

if len(sft_records) < N_SFT:
    raise RuntimeError(
        f"Not enough SFT records: got {len(sft_records)}, need {N_SFT}. "
        f"Try loosening SFT_MAX_TOTAL_TOK / SFT_MAX_TARGET_TOK."
    )

if len(calib_records) < N_CALIB:
    raise RuntimeError(
        f"Not enough calib records: got {len(calib_records)}, need {N_CALIB}. "
        f"Try loosening CALIB_MAX_TOTAL_TOK / CALIB_MAX_TARGET_TOK."
    )

random.shuffle(sft_records)
random.shuffle(calib_records)

sft_path = os.path.join(OUT_DIR, f"sft_{N_SFT}.jsonl")
calib_path = os.path.join(OUT_DIR, f"calib_{N_CALIB}.jsonl")

write_jsonl(sft_path, sft_records[:N_SFT])
write_jsonl(calib_path, calib_records[:N_CALIB])

print("\nDone.")
print("SFT   :", sft_path, len(sft_records[:N_SFT]))
print("Calib :", calib_path, len(calib_records[:N_CALIB]))
print("Calib bucket counts:", {k: len(v) for k, v in calib_buckets.items()})

In [ ]:
def summarize_lengths(records: List[Dict], name: str):
    totals = sorted(r["meta"]["total_tokens"] for r in records)
    targets = sorted(r["meta"]["target_tokens"] for r in records)

    def pct(arr, p):
        idx = min(len(arr) - 1, int(len(arr) * p))
        return arr[idx]

    print(f"\n{name} length summary:")
    print(f"  total_tokens  p50={pct(totals, 0.50)} p90={pct(totals, 0.90)} p95={pct(totals, 0.95)} max={totals[-1]}")
    print(f"  target_tokens p50={pct(targets, 0.50)} p90={pct(targets, 0.90)} p95={pct(targets, 0.95)} max={targets[-1]}")

summarize_lengths(sft_records[:N_SFT], "SFT")
summarize_lengths(calib_records[:N_CALIB], "Calib")

print("\nSample SFT record:")
print(json.dumps(sft_records[0], ensure_ascii=False, indent=2)[:1200])

print("\nSample Calib record:")
print(json.dumps(calib_records[0], ensure_ascii=False, indent=2)[:1200])